[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/comet-ml/opik-examples/blob/main/guides/tracing_three_ways/tracing_three_ways.ipynb)

# Three ways to get traces into Opik

Everything in Opik builds on traces. Each cell below logs the same small support-agent flow, a different way:

| Cell | Way in | How it reaches Opik |
|---|---|---|
| 1 | **Python SDK**: `@track` on your own functions | Opik SDK → REST API |
| 2 | **Integration**: `track_anthropic` wraps the Anthropic client | Opik SDK → REST API |
| 3 | **OpenTelemetry**: plain OTel spans, no Opik SDK in the hot path | OTLP/HTTP → `/api/v1/private/otel` |

All three land in the same project, tagged `sdk`, `integration` and `otel`, as the same trace model:
a **trace**, its **spans** (`llm`, `tool`, `general`, `guardrail`) and, for the chat, a **thread**.

## Setup

In [ ]:
%pip install --quiet --upgrade opik anthropic opentelemetry-sdk opentelemetry-exporter-otlp-proto-http

In [ ]:
import getpass
import os
import uuid

import opik
from opik import opik_context, track

if not os.environ.get("OPIK_API_KEY"):
    # Prompts for your API key and workspace; use opik.configure(use_local=True) for self-hosted.
    opik.configure()

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

OPIK_PROJECT_NAME = os.environ.get("OPIK_PROJECT_NAME", "tracing-three-ways")
# WHY: OPIK_EXAMPLES_MODEL is a litellm-style "provider/model" name; the Anthropic client takes the bare id.
MODEL = os.environ.get("OPIK_EXAMPLES_MODEL", "anthropic/claude-haiku-5-5").removeprefix("anthropic/")
print(f"Logging to Opik project {OPIK_PROJECT_NAME!r} with model {MODEL!r}")

## 1. Python SDK: `@track`

Decorate any function and each call becomes a span; the outermost call is the trace, and nested calls build the tree.
The LLM step is simulated here, so this cell makes no model call: `opik_context` attaches the model and token usage,
and Opik works out the cost.

In [ ]:
@track(type="tool")
def search_kb(query: str) -> list[str]:
    return [
        "Refunds are processed within 5 business days.",
        "Refunds go back to the original payment method.",
    ]


@track(type="llm")
def generate_answer(question: str, docs: list[str]) -> str:
    opik_context.update_current_span(
        model=MODEL,
        provider="anthropic",
        usage={"prompt_tokens": 120, "completion_tokens": 30, "total_tokens": 150},
    )
    return "Refunds take up to 5 business days and go back to your original payment method."


@track(name="support-agent", tags=["sdk"], project_name=OPIK_PROJECT_NAME)
def support_agent(question: str) -> str:
    docs = search_kb(question)
    return generate_answer(question, docs)


print(support_agent("How long do refunds take?"))
opik.flush_tracker()

## 2. Integration: `track_anthropic`

One line wraps the Anthropic client and every call is logged as an `llm` span with model, tokens and cost.
The same pattern covers `track_openai`, `track_bedrock`, `OpikTracer` for LangChain/LangGraph, and more.
Integrations are built on the SDK, so their spans nest under your own `@track` functions.
Setting a `thread_id` groups the two turns into one conversation in the **Threads** tab.

In [ ]:
import anthropic
from opik.integrations.anthropic import track_anthropic

client = track_anthropic(anthropic.Anthropic())


@track(name="support-chat", tags=["integration"], project_name=OPIK_PROJECT_NAME)
def chat_turn(message: str, history: list[dict], thread_id: str) -> str:
    opik_context.update_current_trace(thread_id=thread_id)
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system="You are a concise customer support agent for an online store. Answer in two sentences.",
        messages=[*history, {"role": "user", "content": message}],
    )
    return response.content[0].text


thread_id = f"demo-chat-{uuid.uuid4().hex[:8]}"
history: list[dict] = []
for message in ["How long do refunds take?", "And if I paid by credit card?"]:
    reply = chat_turn(message, list(history), thread_id)
    history += [{"role": "user", "content": message}, {"role": "assistant", "content": reply}]
    print(f"> {message}\n{reply}\n")

opik.flush_tracker()

## 3. OpenTelemetry

No Opik SDK on the sending side: a standard OTel `TracerProvider` exports OTLP/HTTP to Opik's endpoint, so the
same works from Java, .NET, Go or Ruby. Opik reads the `gen_ai.*` semantic-convention attributes to type the spans
(`gen_ai.request.model` makes an `llm` span, `gen_ai.tool.call.*` a `tool` span). In a real app an OTel
instrumentation library or framework sets them for you.

The usual setup is two environment variables:
```
OTEL_EXPORTER_OTLP_ENDPOINT=<opik-url>/api/v1/private/otel
OTEL_EXPORTER_OTLP_HEADERS=Authorization=<api-key>,Comet-Workspace=<workspace>,projectName=<project>
```
Here they are built from your Opik config so the cell runs as-is.

In [ ]:
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

config = opik.config.OpikConfig()
headers = {
    "Authorization": config.api_key,
    "Comet-Workspace": config.workspace,
    "projectName": OPIK_PROJECT_NAME,
}
exporter = OTLPSpanExporter(
    endpoint=f"{config.url_override.rstrip('/')}/v1/private/otel/v1/traces",
    headers={key: value for key, value in headers.items() if value},
)
provider = TracerProvider()
provider.add_span_processor(BatchSpanProcessor(exporter))
tracer = provider.get_tracer("tracing-three-ways")

question = "How long do refunds take?"
answer = "Refunds take up to 5 business days."

with tracer.start_as_current_span("support-agent-otel") as root:
    root.set_attribute("input", question)
    root.set_attribute("opik.tags", ["otel"])

    with tracer.start_as_current_span("search_kb") as tool_span:
        tool_span.set_attribute("gen_ai.tool.call.arguments", '{"query": "refunds"}')
        tool_span.set_attribute("gen_ai.tool.call.result", "Refunds are processed within 5 business days.")

    with tracer.start_as_current_span(f"chat {MODEL}") as llm_span:
        llm_span.set_attribute("gen_ai.system", "anthropic")
        llm_span.set_attribute("gen_ai.request.model", MODEL)
        llm_span.set_attribute("gen_ai.usage.input_tokens", 120)
        llm_span.set_attribute("gen_ai.usage.output_tokens", 30)
        llm_span.set_attribute("input", question)
        llm_span.set_attribute("output", answer)

    root.set_attribute("output", answer)

# WHY: the OTel exporter reports failures (e.g. a 401 from a bad API key) only as a log line, not an exception.
provider.force_flush()
print("Flushed OTel trace 'support-agent-otel'; check the output above for export errors.")

## See it in Opik

Open the project and compare the traces tagged `sdk`, `integration` and `otel`: the same trace tree whichever
way they arrived. The `integration` traces also appear as one conversation under **Threads**.